In [1]:
import pandas as pd
import numpy as np
import yfinance as yf
import requests 
import os
import logging
import time
from tqdm import tqdm
import pyarrow as pa
import pyarrow.parquet as pq

# Adjust path if running from notebooks directory
if os.getcwd().endswith('notebooks'):
    os.chdir('..')

from src.data_loader import FinancialDataLoader

In [ ]:
# Globals

raw_data_dir = "data/raw"
proc_data_dir = "data/processed"

# Required for SEC access
headers = {
    "User-Agent": "FIRST LAST EMAIL"
}

In [3]:
# Setting up logger
logger = logging.getLogger(__name__)
logging.basicConfig(level=logging.INFO, format = "%(asctime)s - %(levelname)s - %(message)s")

loader = FinancialDataLoader()

## Creating Datasets

In [ ]:
# File downloader wrapper
def download_file(url: str, filename: str, headers: dict = {}) -> bool:
    return loader.download_file(url, filename, headers)

In [ ]:
download_file("https://www.sec.gov/files/company_tickers.json","company_tickers.json", headers)

In [4]:
df = pd.read_json("data/raw/company_tickers.json", orient = "index")
df["cik_str"] = df["cik_str"].astype(str).str.zfill(10)

display(df.head())

,cik_str,ticker,title
0,0001045810,NVDA,NVIDIA CORP
1,0001652044,GOOGL,Alphabet Inc.
2,0000320193,AAPL,Apple Inc.
3,0000789019,MSFT,MICROSOFT CORP
4,0001018724,AMZN,AMAZON COM INC


In [5]:
# Metrics fetcher wrapper
def fetch_single_comp_metrics(ticker: str) -> dict:
    return loader._fetch_single_comp_metrics(ticker)

In [6]:
# Testing function
fetch_single_comp_metrics("NVDA")

{'ticker': 'NVDA',
 'enterprise_value': 5170627870720,
 'forwardPE': 17.027882,
 'ev_to_ebitda': 31.24,
 'ebitda': 165514002432,
 'total_cash': 53171998720,
 'total_debt': 12814000128,
 'employee_count': 42000,
 'estimated_revenue': 253491003392,
 'sector': 'Technology',
 'industry': 'Semiconductors',
 'business_summary': "NVIDIA Corporation operates as a data center scale AI infrastructure company. The company operates through two segments, Compute & Networking, and Graphics segments. The Compute & Networking segment provides data center accelerated computing and networking platforms and artificial intelligence solutions and software, and automotive platforms and autonomous and electric vehicle solutions, including software. The Graphics segment offers GeForce GPUs for gaming and PCs; Quadro/NVIDIA RTX GPUs for enterprise workstation graphics. The company's products are used in gaming, professional visualization, data center, and automotive markets. The company sells its products to o

In [12]:
# Master Table Builder wrapper
def build_csv_comps_table(raw_data_path: str, output_csv: str, chunk_size: int = 50, limit: int = None):
    return loader.build_raw_master_table(raw_data_path, output_csv, chunk_size, limit)

In [13]:
build_csv_comps_table("data/raw/company_tickers.json", "data/raw/master_metrics.csv", 50, limit=10)

2026-05-26 00:09:15,730 - INFO - Starting large data pull...
2026-05-26 00:09:15,765 - INFO - Test Mode: Only processing the first 10 companies


True

In [14]:
pd.read_csv("data/raw/master_metrics.csv")

,ticker,enterprise_value,forwardPE,ev_to_ebitda,ebitda,total_cash,total_debt,employee_count,estimated_revenue,sector,industry,business_summary
0,NVDA,5170627870720,17.027882,31.240,165514002432,53171998720,12814000128,42000,253491003392,Technology,Semiconductors,NVIDIA Corporation operates as a data center s...
1,GOOGL,4609100742656,26.421965,28.572,161315995648,126839996416,95875997696,194668,422498009088,Communication Services,Internet Content & Information,Alphabet Inc. offers various products and plat...
2,AAPL,4551953350656,32.155922,28.454,159975997440,68507000832,84710998016,166000,451442016256,Technology,Consumer Electronics,"Apple Inc. designs, manufactures, and markets ..."
3,MSFT,3156523876352,21.645250,17.113,184457003008,78227996672,125431996416,228000,318272995328,Technology,Software - Infrastructure,Microsoft Corporation develops and supports so...
4,AMZN,2957284474880,27.011812,18.974,155860992000,143088992256,235540004864,1575000,742775980032,Consumer Cyclical,Internet Retail,"Amazon.com, Inc. engages in the retail sale of..."
5,AVGO,2012698509312,22.681368,54.079,37218000896,14174000128,66056998912,33000,68281999360,Technology,Semiconductors,"Broadcom Inc. designs, develops, and supplies ..."
6,META,1554687197184,16.948385,14.223,109308002304,81180000256,86769000448,77986,214962995200,Communication Services,Internet Content & Information,"Meta Platforms, Inc. engages in the developmen..."
7,TSLA,1571808870400,169.746060,141.681,11093999616,44743000064,15889999872,134785,97878999040,Consumer Cyclical,Auto Manufacturers,"Tesla, Inc. designs, develops, manufactures, l..."
8,BRK-B,-265528623104,22.827251,-2.250,118003998720,397383008256,128885997568,387800,375394009088,Financial Services,Insurance - Diversified,"Berkshire Hathaway Inc., together with its sub..."
9,WMT,1028526637056,36.567010,22.847,45019000832,10728999936,74179002368,2100000,725304999936,Consumer Defensive,Discount Stores,Walmart Inc. engages in the operation of retai...


In [15]:
if os.path.exists("data/processed/missed_tickers.csv"):
    df_missed = pd.read_csv("data/processed/missed_tickers.csv")
    print(df_missed["failed_tickers"].tolist())

In [18]:
# Creating ML-Ready Tables (Leak-Free Architecture)
master_file = "data/raw/master_metrics.csv"

loader.build_public_training_table(master_file, "data/processed/PUBLIC_training.parquet")
loader.build_private_training_table(master_file, "data/processed/PRIVATE_training.parquet")

2026-05-26 00:12:23,672 - INFO - Building Public Engine Training Table...
2026-05-26 00:12:23,709 - INFO - Public Training Table complete: 10 rows.
2026-05-26 00:12:23,709 - INFO - Building Private Engine Training Table...
2026-05-26 00:12:23,716 - INFO - Private Training Table complete: 10 rows.


True

In [3]:
print("Public Sample:")
display(pd.read_parquet("data/processed/PUBLIC_training.parquet").head())

print("Private Sample:")
display(pd.read_parquet("data/processed/PRIVATE_training.parquet").head())

Public Sample:


,ticker,enterprise_value,forwardPE,ev_to_ebitda,ebitda,total_cash,total_debt,sector,business_summary
0,NVDA,5.170628e+12,17.027882,31.240,165514002432,5.317200e+10,1.281400e+10,Technology,NVIDIA Corporation operates as a data center s...
1,GOOGL,4.609101e+12,26.421965,28.572,161315995648,1.268400e+11,9.587600e+10,Communication Services,Alphabet Inc. offers various products and plat...
2,AAPL,4.551953e+12,32.155922,28.454,159975997440,6.850700e+10,8.471100e+10,Technology,"Apple Inc. designs, manufactures, and markets ..."
3,MSFT,3.156524e+12,21.645250,17.113,184457003008,7.822800e+10,1.254320e+11,Technology,Microsoft Corporation develops and supports so...
4,AMZN,2.957284e+12,27.011812,18.974,155860992000,1.430890e+11,2.355400e+11,Consumer Cyclical,"Amazon.com, Inc. engages in the retail sale of..."


Private Sample:


,ticker,enterprise_value,employee_count,estimated_revenue,sector,business_summary
0,NVDA,5.170628e+12,42000.0,2.534910e+11,Technology,NVIDIA Corporation operates as a data center s...
1,GOOGL,4.609101e+12,194668.0,4.224980e+11,Communication Services,Alphabet Inc. offers various products and plat...
2,AAPL,4.551953e+12,166000.0,4.514420e+11,Technology,"Apple Inc. designs, manufactures, and markets ..."
3,MSFT,3.156524e+12,228000.0,3.182730e+11,Technology,Microsoft Corporation develops and supports so...
4,AMZN,2.957284e+12,1575000.0,7.427760e+11,Consumer Cyclical,"Amazon.com, Inc. engages in the retail sale of..."
